In [151]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
import spacy
from textblob import TextBlob
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
import faiss

#### Data overview

In [153]:
df_books = pd.read_csv("../data/books_data.csv")#, nrows=100)
df_books.head(2)

,Title,description,authors,image,previewLink,publisher,publishedDate,infoLink,categories,ratingsCount
0,Its Only Art If Its Well Hung!,NaN,['Julie Strain'],http://books.google.com/books/content?id=DykPA...,http://books.google.nl/books?id=DykPAAAACAAJ&d...,NaN,1996,http://books.google.nl/books?id=DykPAAAACAAJ&d...,['Comics & Graphic Novels'],NaN
1,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,['Philip Nel'],http://books.google.com/books/content?id=IjvHQ...,http://books.google.nl/books?id=IjvHQsCn_pgC&p...,A&C Black,2005-01-01,http://books.google.nl/books?id=IjvHQsCn_pgC&d...,['Biography & Autobiography'],NaN


In [154]:
df_reviews = pd.read_csv("../data/Books_rating.csv")#, nrows=100)
df_reviews = df_reviews.sample(frac=0.001)
df_reviews.head(5)

,Id,Title,Price,User_id,profileName,score,time,summary,text
1724835,0380800845,Jackie & Me (Baseball Card Adventures),5.99,A1QKQV0FSIFPCN,Lisa Ard,4.0,1341705600,Historical fiction mixed with fun adventure,Joe Stoshack travels through time using a base...
1567191,B000K6KBDA,Starship Troopers,NaN,NaN,NaN,5.0,885859200,the best sci-fi space opera book ever,If you're not intelligent enough to get the se...
1179148,B00085VDKI,"The Count of Monte Cristo, (The Rittenhouse cl...",NaN,A18SSBZAYK5IN3,Gabriel Hall,1.0,1354147200,Poor Quality,Let me be clear that I am a big fan of The Cou...
311407,B000G84FT8,We Caught The Fire,NaN,A1CZJNCRZTHTKC,naplesdude,5.0,1352764800,awesome story,This book has changed my family's life and cou...
1133670,B000NHNM3C,George Orwell 1984,NaN,A17JRC494S96GM,Zefram Cochrane,1.0,1248307200,"Review of this digital download, not of the ma...","Ok, just where to begin. First this digital do..."


In [155]:
df_reviews.info()

<class 'pandas.DataFrame'>
Index: 3000 entries, 1724835 to 1671786
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Id           3000 non-null   str    
 1   Title        2999 non-null   str    
 2   Price        530 non-null    float64
 3   User_id      2462 non-null   str    
 4   profileName  2462 non-null   str    
 5   score        3000 non-null   float64
 6   time         3000 non-null   int64  
 7   summary      3000 non-null   str    
 8   text         3000 non-null   str    
dtypes: float64(2), int64(1), str(6)
memory usage: 234.4 KB


In [156]:
df_merged = df_reviews.merge(df_books[["Title", "authors", "publishedDate", "categories"]], on="Title", how="left")
df_merged.columns

Index(['Id', 'Title', 'Price', 'User_id', 'profileName', 'score', 'time',
       'summary', 'text', 'authors', 'publishedDate', 'categories'],
      dtype='str')

#### General analysis

In [143]:
# avg score per book

# df_reviews.groupby("Title")["score"].mean().sort_values(ascending=False).head(10)
df_reviews.groupby("Title").agg(
    avg_score=("score", "mean"),
    n_reviews=("score", "count")
).sort_values(
    by=["avg_score", "n_reviews"],
    ascending=[False, False]
).head(10)

,avg_score,n_reviews
Title,,
How to Win Friends & Influence People,5.0,13
"The China Study: The Most Comprehensive Study of Nutrition Ever Conducted and the Startling Implications for Diet, Weight Loss and Long-term Health",5.0,12
All Creatures Great and Small,5.0,10
The Stranger Beside Me,5.0,9
A Christmas Carol (Enriched Classics (Pocket)),5.0,8
A Christmas Carol [One Voice Recordings Edition],5.0,7
"A Tale of Two Cities, Literary Touchstone Edition",5.0,7
Batman: Year One,5.0,7
"Huckleberry Finn,",5.0,7


In [144]:
# avg score per author

df_merged.groupby("authors").agg(
    avg_score=("score", "mean"),
    n_reviews=("score", "count"),
    n_books=("Title", "nunique")
).sort_values(
    by=["avg_score", "n_reviews", "n_books"],
    ascending=[False, False, False]
).head(10)

,avg_score,n_reviews,n_books
authors,,,
['Dr. Seuss'],5.0,29,12
['Brian Jacques'],5.0,12,4
['FREDERICK DOUGLASS'],5.0,11,3
['Hannah Hurnard'],5.0,10,3
['Robert McCloskey'],5.0,9,3
['Jon Stone'],5.0,8,2
['A. W. Tozer'],5.0,7,3
['Betty Crocker'],5.0,7,3
"['Richard Atwater', 'Florence Atwater']",5.0,7,2


In [145]:
# avg score per genre

df_merged.groupby("categories").agg(
    avg_score=("score", "mean"),
    n_reviews=("score", "count"),
    n_books=("Title", "nunique")
).sort_values(
    by=["avg_score", "n_reviews", "n_books"],
    ascending=[False, False, False]
).head(10)

,avg_score,n_reviews,n_books
categories,,,
['American wit and humor'],5.0,8,6
['Humorous stories'],5.0,8,3
['Mexico'],5.0,6,3
['Business ethics'],5.0,6,2
['Audio-visual education'],5.0,6,1
['Cats'],5.0,5,5
"['Comic books, strips, etc']",5.0,5,5
['Conduct of life'],5.0,5,4
['Reading (Elementary)'],5.0,5,2


In [146]:
# books with the most reviews

df_reviews.groupby("Title").agg(
    n_reviews=("score", "count"),
    avg_score=("score", "mean")
).sort_values("n_reviews", ascending=False).head(10)

,n_reviews,avg_score
Title,,
The Hobbit,219,4.652968
Pride and Prejudice,192,4.421875
Atlas Shrugged,125,3.888000
Wuthering Heights,121,4.041322
The Giver,83,4.385542
Of Mice and Men,80,4.525000
Great Expectations,78,4.307692
Harry Potter and The Sorcerer's Stone,70,4.700000
Mere Christianity,66,4.606061


In [147]:
# users with most reviews

df_reviews.groupby("profileName").agg(
    n_reviews=("score", "count"),
    avg_score=("score", "mean")
).sort_values("n_reviews", ascending=False).head(10)

,n_reviews,avg_score
profileName,,
Midwest Book Review,50,5.000000
A Customer,43,4.000000
"E. A Solinas ""ea_solinas""",36,4.750000
Harriet Klausner,34,4.558824
John,22,4.363636
"Shalom Freedman ""Shalom Freedman""",18,4.500000
"bernie ""xyzzy""",16,5.000000
"Donald Mitchell ""Jesus Loves You!""",15,4.333333
Charles Ashbacher,14,4.500000


In [148]:
# correlation between price, score and number of reviews

df_comp = df_merged[['Price', 'score']]
df_comp['n_reviews'] = df_merged.groupby("Title")["score"].transform("count")

df_comp[['Price', 'score', 'n_reviews']].corr()

,Price,score,n_reviews
Price,1.000000,-0.012413,0.008660
score,-0.012413,1.000000,0.031225
n_reviews,0.008660,0.031225,1.000000


#### NLP Preprocesssing

In [157]:
nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    doc = nlp(text.lower())
    tokens = [t.lemma_ for t in doc if not t.is_stop and t.is_alpha]
    return " ".join(tokens)

df_reviews["clean_review"] = df_reviews["text"].apply(clean_text)

In [158]:
df_reviews.sample(10)[["text", "clean_review"]]

,text,clean_review
2310214,"This is a brilliant, heart-felt, humorous yet ...",brilliant heart feel humorous introduction soc...
2538032,1000+ pages is very intimidating. I know; some...,page intimidating know suggest thought read fo...
2381087,I love this story; I hate those times when peo...,love story hate time people judgemental guess ...
1672548,Brock's supposed 'turnaround' is questionable ...,brock suppose turnaround questionable well lat...
2485010,"Ho hum, still waiting for a publisher rewrite ...",ho hum wait publisher rewrite poor kindle vers...
1654124,Williams is the father of the revisionist myth...,williams father revisionist myth go wrong worl...
718583,This books is not helpful if you are an impend...,book helpful impending mother want balanced in...
704800,"From the Publisher:""Alison Dundes Renteln's ""T...",dunde renteln cultural defense extraordinarily...
741069,"After two false starts, Robin Pilcher has come...",false start robin pilcher come new novel risk ...
1311433,I am slowly working my way through his bibliog...,slowly work way bibliography up down pratchett...


#### Sentiment Analysis

In [159]:
df_reviews["sentiment"] = df_reviews["clean_review"].apply(
    lambda x: TextBlob(x).sentiment.polarity
)

In [160]:
# books with highest average sentiment

title_sentiment = df_reviews.groupby("Title")["sentiment"].mean()
title_sentiment.sort_values(ascending=False).head(10)

Title
Research Strategies: Finding Your Way Through the Information Fog                       1.000000
Internet Performance Survival Guide: QoS Strategies for Multiservice Networks           1.000000
Favoured Child (Wideacre Trilogy 2)                                                     1.000000
Prince of Dreams: A Tale of Tristan and Essylte                                         1.000000
Exodus from Obesity: 2nd Edition                                                        1.000000
My Friends' Secrets:Conversations with My Friends about Beauty, Health and Happiness    1.000000
Disposable: A History of Skateboard Art                                                 1.000000
The Tortilla Curtain                                                                    1.000000
The Hart Brothers Simon & Callaghan: Beloved\Callaghan's Bride                          0.933333
Joy in Our Weakness: A Gift of Hope from the Book of Revelation                         0.933333
Name: sentiment, dtype: 

In [166]:
# best scored authors

df_merged_2 = df_reviews.merge(df_books[["Title", "authors", "categories"]], on="Title", how="left")
author_sentiment = df_merged_2.groupby("authors")["sentiment"].mean()
author_sentiment.sort_values(ascending=False).head(10)

authors
['Marsha Miller']           1.000000
['T. Coraghessan Boyle']    1.000000
['Joan Collins']            1.000000
['Geoff Huston']            1.000000
['Sean Cliver']             1.000000
['Nancy McKenzie']          1.000000
['William B. Badke']        1.000000
['Marva J. Dawn']           0.933333
['Diana Palmer']            0.933333
['Naoko Takeuchi']          0.850000
Name: sentiment, dtype: float64

In [167]:
# best scored authors

df_merged_2['sentiment'] = df_merged_2['sentiment']*5

df_merged_2.groupby("authors").agg(
    avg_score=("score", "mean"),
    avg_sentiment=("sentiment", "mean"),
    n_reviews=("score", "count")
).sort_values(
    by=["avg_sentiment", "avg_score", "n_reviews"],
    ascending=[False, False, False]
).head(10)

,avg_score,avg_sentiment,n_reviews
authors,,,
['Geoff Huston'],5.0,5.000000,1
['Joan Collins'],5.0,5.000000,1
['Nancy McKenzie'],5.0,5.000000,1
['Sean Cliver'],5.0,5.000000,1
['T. Coraghessan Boyle'],5.0,5.000000,1
['William B. Badke'],5.0,5.000000,1
['Marsha Miller'],2.0,5.000000,1
['Diana Palmer'],5.0,4.666667,1
['Marva J. Dawn'],5.0,4.666667,1


#### Semantic search

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

df_reviews["embedding"] = df_reviews["clean_review"].apply(
    lambda x: model.encode(x)
)

embeddings = np.vstack(df_reviews["embedding"].values)

index = faiss.IndexFlatL2(384)
index.add(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6552.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [57]:
def search_reviews(query, top_k=10):
    query_vector = model.encode([query])
    D, I = index.search(query_vector, top_k)
    return D, I

In [60]:
# search_query = "I love this book"
search_query = "reviews that deeply analyze character development"

In [98]:
D, I = search_reviews(search_query, top_k=5)
for distance, idx in zip(D[0], I[0]):
    print("Review relevance:", distance)
    print("Review:", df_reviews.iloc[idx]["text"])
    print("User:", df_reviews.iloc[idx]["profileName"])
    print()

Review relevance: 1.0500581
Review: When I first read this the I was mezmerized at the style of writting. I soon became so hooked on this book that I wanted to fall on my knees and shut out the rest of the world. Haddon delivers a portrayal of a woman that is hurt loney and abused and I think she does a wonderful job making the reader feel like the character.
User: Clara

Review relevance: 1.0920141
Review: It's glaringly obvious that all of the glowing reviews have been written by the same person, perhaps the author herself. They all have the same misspellings and poor sentence structure that is featured in the book. Who made Veronica Haddon think she is an author?
User: Bronx Girl

Review relevance: 1.1323907
Review: I purchased this book after reading rave reviews of it included within a review of a really good book I'd read. Apparently, the author (or her friends and family) are placing reviews of good books and encouraging people to buy this book if you liked the one they were rev

#### Users clustering

In [107]:
model = SentenceTransformer('all-MiniLM-L6-v2')

df_reviews["embedding"] = df_reviews["clean_review"].apply(
    lambda x: model.encode(x)
)

embeddings = np.vstack(df_reviews["embedding"].values)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7339.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [108]:
kmeans = KMeans(n_clusters=8)
df_reviews["cluster"] = kmeans.fit_predict(embeddings)

In [111]:
df_reviews[['profileName','cluster']].sort_values("cluster")

,profileName,cluster
96,"Kiki B. ""The Shepherdess""",0
97,Heddlemaid,0
98,"Linda Pool ""cookbookaholic""",0
92,Gary W. Marian,0
91,NaN,0
...,...,...
11,Dr. Terry W. Dorsett,7
56,"Nov 10 ""Tom""",7
49,GodsBreath.wordpress,7
48,haskell,7
